# Estimating $\kappa$ and $\tau$ from patient fiber fields

Simulated-moment fit of the SPDE prior parameters (`bayes_cep.statistics.prior_moment_fit`): for each pair $(\kappa, \tau)$ of a grid, prior samples on the mesh are compared with the patient field through two statistics that do not change under $\theta \to \theta + \pi$: the correlation $\overline{\cos 2\Delta}$ of simplex pairs binned by their distance, and the concentration $R = |\overline{e^{2i\theta}}|$. The loss is the squared deviation of the statistics of the samples from those of the field (see the module docstring).

The grid scores are computed by `scripts/fit_prior_parameters.py` (about 3 minutes per patient for the raw fields, 8 minutes for the smoothed ones) and stored in `patient_data/prior_fits/`:

```
pixi run python scripts/fit_prior_parameters.py --smoothing-scale 0   # raw fields
pixi run python scripts/fit_prior_parameters.py --smoothing-scale 3   # smoothed fields
```

For a smoothed field, the prior samples are smoothed with the same scale before the statistics are computed, so that the roughness a Bilaplacian prior cannot produce is removed from both sides.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

from bayes_cep.statistics.prior_moment_fit import MomentFitResult, pool_losses
from single_runs.reference import MOMENT_FIT_NUM_FITTED_BINS

PATIENT_DATA = Path.cwd().parent / "patient_data"
VERSION = "raw"  # "raw" or "smoothed_scale_3"
PATIENTS = [f"patient_0{i}" for i in range(1, 8)]
HELD_OUT = "patient_01"  # the out-of-sample patient of the last section

results = {
    patient: MomentFitResult.load(PATIENT_DATA / "prior_fits" / VERSION / f"{patient}.npz")
    for patient in PATIENTS
}
kappas, taus = results[PATIENTS[0]].kappas, results[PATIENTS[0]].taus
print(f"{VERSION}: grid of {kappas.size} kappa x {taus.size} tau values")

To compute a fit interactively instead (here for one patient and a small grid), run the cell below; it uses the same function as the script.

In [ ]:
RUN_EXAMPLE = False
if RUN_EXAMPLE:
    from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
    from bayes_cep.mesh.io import load_pyvista_mesh
    from bayes_cep.mesh.topology import get_triangle_connectivity
    from bayes_cep.preprocessing.angle_field import compute_angle_from_fiber_vectors
    from bayes_cep.statistics.prior_moment_fit import fit_prior_parameters
    from single_runs.reference import reference_moment_fit_settings

    patient_dir = PATIENT_DATA / "patient_01"
    mesh = load_pyvista_mesh(patient_dir / "mesh.vtu")
    basis_vectors = np.load(patient_dir / "basis_vecs.npy")
    observed = compute_angle_from_fiber_vectors(
        np.load(patient_dir / "fiber_field.npy"), basis_vectors[..., 0], basis_vectors[..., 1]
    )
    example = fit_prior_parameters(
        mesh,
        observed,
        (0.2, 0.6),
        (0.45, 1.4),
        reference_moment_fit_settings(0.0),
        NearestNeighborInterpolationStrategy(),
    )
    print(example.loss, example.best_parameters())

## Loss surfaces per patient

The loss over the grid (log color scale), with the minimum marked. A long dark valley instead of a point means that the data fix a combination of $\kappa$ and $\tau$ but not both.

In [ ]:
def plot_surface(axis, loss, title, markers=()):
    image = axis.imshow(loss, origin="lower", cmap="viridis", norm=LogNorm(), aspect="auto")
    axis.set_xticks(range(taus.size), [f"{t:g}" for t in taus], fontsize=7)
    axis.set_yticks(range(kappas.size), [f"{k:g}" for k in kappas], fontsize=7)
    axis.set_xlabel("tau")
    axis.set_ylabel("kappa")
    axis.set_title(title, fontsize=9)
    for (row, column), style, label in markers:
        axis.plot(column, row, style, markersize=10, label=label)
    return image


fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for axis, patient in zip(axes.ravel(), PATIENTS, strict=False):
    result = results[patient]
    best = np.unravel_index(np.argmin(result.loss), result.loss.shape)
    image = plot_surface(
        axis, result.loss, f"{patient}: min {result.loss.min():.3f}", [(best, "r*", "minimum")]
    )
axes.ravel()[-1].axis("off")
fig.colorbar(image, ax=axes, label="loss")

In [ ]:
print(f"{'patient':<12}{'kappa':>7}{'tau':>6}{'kappa*tau':>11}{'loss':>8}   within 2x of the minimum: kappa range, kappa*tau range")
for patient, result in results.items():
    kappa, tau = result.best_parameters()
    near = result.loss <= 2 * result.loss.min()
    rows, columns = np.nonzero(near)
    product = kappas[rows] * taus[columns]
    print(
        f"{patient:<12}{kappa:>7g}{tau:>6g}{kappa * tau:>11.2f}{result.loss.min():>8.3f}   "
        f"{kappas[rows].min():g}-{kappas[rows].max():g}, {product.min():.2f}-{product.max():.2f}"
    )

## How well does the best pair reproduce the statistics?

The correlation versus distance of the field (black) and of the samples of the best pair (red), for each patient. The shaded bins are not in the loss (the outer two bins).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
for axis, patient in zip(axes.ravel(), PATIENTS, strict=False):
    result = results[patient]
    row, column = np.unravel_index(np.argmin(result.loss), result.loss.shape)
    edges = result.distance_bin_edges
    bins = np.arange(edges.size - 1)
    axis.plot(bins, result.observed_correlation, "ko-", label=f"field, R {result.observed_concentration:.2f}")
    axis.plot(bins, result.correlation[row, column], "r^-", label=f"samples, R {result.concentration[row, column]:.2f}")
    axis.axvspan(MOMENT_FIT_NUM_FITTED_BINS - 0.5, bins[-1] + 0.5, color="gray", alpha=0.15)
    axis.set_xticks(bins, [f"{lo:g}" for lo in edges[:-1]], fontsize=7)
    axis.set_title(f"{patient}: kappa {kappas[row]:g}, tau {taus[column]:g}", fontsize=9)
    axis.set_xlabel("centroid distance, lower bin edge")
    axis.legend(fontsize=7)
axes[0, 0].set_ylabel("mean cos 2 Delta")
axes.ravel()[-1].axis("off")
fig.tight_layout()

## Out-of-sample estimate for one patient

Each patient has its own surface, so the question is whether the other patients predict the parameters of the held-out one. The other patients' loss surfaces are summed on the common grid; the minimum of the sum is the single pair that fits the ensemble best (`pool_losses`). Its quality for the held-out patient is the held-out loss at that pair, compared with the minimum of the patient's own surface and with the rank of the pair among the grid values. The sum assumes that all patients share one pair, which the table of the best pairs above tests.

In [ ]:
def out_of_sample(held_out):
    others = [results[p] for p in PATIENTS if p != held_out]
    pooled = pool_losses(others)
    row, column = np.unravel_index(np.argmin(pooled), pooled.shape)
    own = results[held_out].loss
    rank = int(np.sum(own < own[row, column])) + 1
    return (row, column), pooled, own[row, column], own.min(), rank


print(f"{'held out':<12}{'pooled (kappa, tau)':>22}{'own loss there':>16}{'own minimum':>13}{'ratio':>7}{'rank / ' + str(kappas.size * taus.size):>10}")
for patient in PATIENTS:
    (row, column), _, loss_there, loss_min, rank = out_of_sample(patient)
    print(
        f"{patient:<12}{f'({kappas[row]:g}, {taus[column]:g})':>22}{loss_there:>16.3f}"
        f"{loss_min:>13.3f}{loss_there / loss_min:>7.1f}{rank:>10}"
    )

pooled_cell, pooled, *_ = out_of_sample(HELD_OUT)
own_best = np.unravel_index(np.argmin(results[HELD_OUT].loss), results[HELD_OUT].loss.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_surface(axes[0], pooled, f"sum of the other patients' losses", [(pooled_cell, "r*", "pooled minimum")])
image = plot_surface(
    axes[1],
    results[HELD_OUT].loss,
    f"{HELD_OUT} (held out)",
    [(own_best, "w*", "own minimum"), (pooled_cell, "r*", "pooled minimum of the others")],
)
axes[1].legend(fontsize=8)
fig.tight_layout()

## Reading

Fill in after looking at the tables: whether the best pairs of the patients agree (the product $\kappa\tau$ is expected to agree better than $\kappa$), and whether the pooled pair of the other patients has a loss for the held-out patient within a factor of a few of its own minimum and a low rank. If so, the ensemble gives a usable out-of-sample estimate, and the held-out patient can be inferred with the pooled pair; if not, the patients differ too much for a common pair and the estimate has to come from the patient's own field.